# Diários Oficiais do Rio — Downloader

Notebook standalone para baixar automaticamente:

- **DCM** — Diário da Câmara Municipal do Rio de Janeiro · `dcmdigital.camara.rj.gov.br`
- **DOP** — Diário Oficial da Prefeitura do Rio de Janeiro · `doweb.rio.rj.gov.br`

### Modos de uso

| Modo | DCM | DOP |
|---|---|---|
| `hoje` | última edição publicada | última edição publicada |
| `edicao` | por número + ano | por número ou data |
| `range` | intervalo de números | intervalo de números ou datas |

Configure as variáveis na próxima célula e rode **Run All**. Os PDFs são salvos na pasta `DESTINO`.

---
*Parte do projeto [clipping-project](https://github.com/OttoBoop/clipping-project).*


## 1. Configuração

Edite **somente** as variáveis abaixo.

In [ ]:
# ==========================================================
#  O QUE BAIXAR
# ==========================================================
MODO   = "hoje"      # "hoje" | "edicao" | "range"
FONTE  = "ambos"     # "dcm"  | "dop"    | "ambos"

# --- usado se MODO == "edicao" ---
EDICAO_NUMERO = 108        # número da edição (obrigatório pro DCM)
EDICAO_ANO    = 2025
EDICAO_DATA   = None       # alternativa pro DOP: "YYYY-MM-DD" ou None

# --- usado se MODO == "range" ---
RANGE_TIPO   = "edicao"    # "edicao" (DCM e DOP) | "data" (só DOP)
RANGE_INICIO = 156         # número de edição OU data "YYYY-MM-DD"
RANGE_FIM    = 158

# ==========================================================
#  ONDE SALVAR
# ==========================================================
DESTINO = "./downloads"    # ex.: "/content/drive/MyDrive/DOs" no Colab


## 2. Instalação

Descomente as linhas abaixo na primeira vez (Colab / ambiente novo).

In [ ]:
# !pip install --quiet requests beautifulsoup4 PyPDF2 selenium pandas
# !apt-get install -y chromium-chromedriver   # necessário apenas pro DCM


In [ ]:
import os, re, json, time, shutil, datetime
from urllib.parse import urljoin
import requests
from PyPDF2 import PdfReader, PdfMerger
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

os.makedirs(DESTINO, exist_ok=True)
print(f"Destino: {os.path.abspath(DESTINO)}")


## 3. DOP — Diário Oficial da Prefeitura

Fonte: <https://doweb.rio.rj.gov.br>

Implementação 100% HTTP (sem Selenium). A home expõe a edição mais recente em uma variável JavaScript chamada `DADOS_ULTIMA_DATA`, que lista os cadernos do dia (principal + suplementos).

> **Limitação conhecida:** o portal não documenta um endpoint público de arquivo histórico.
> As funções `baixar_dop_por_data` / `baixar_dop_por_edicao` só funcionam para itens que ainda estejam expostos em `DADOS_ULTIMA_DATA`. Para datas antigas, ajuste a função com o endpoint que aparecer no devtools do navegador.


In [ ]:
DOP_BASE = "https://doweb.rio.rj.gov.br"
DOP_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

def _dop_pegar_itens_home(session):
    """Lê DADOS_ULTIMA_DATA do HTML da home e retorna a lista de itens."""
    r = session.get(DOP_BASE, headers=DOP_HEADERS, timeout=30)
    r.raise_for_status()
    m = re.search(r"let DADOS_ULTIMA_DATA = (\{.*?\});", r.text, re.DOTALL)
    if not m:
        raise RuntimeError("Variável DADOS_ULTIMA_DATA não encontrada na home do DOP.")
    return json.loads(m.group(1)).get("itens", [])

def _dop_baixar_pdf(session, item_id, nome_arquivo, destino):
    url = f"{DOP_BASE}/portal/edicoes/download/{item_id}"
    print(f"[DOP] baixando id={item_id} <- {url}")
    r = session.get(url, headers=DOP_HEADERS, timeout=120)
    r.raise_for_status()
    if "application/pdf" not in r.headers.get("Content-Type", "").lower():
        raise RuntimeError(f"Resposta não é PDF (Content-Type={r.headers.get('Content-Type')})")
    caminho = os.path.join(destino, nome_arquivo)
    with open(caminho, "wb") as f:
        f.write(r.content)
    print(f"[DOP] OK {caminho} ({len(r.content)/1024/1024:.2f} MB)")
    return caminho

def baixar_dop_hoje(destino=None):
    """Baixa a edição mais recente publicada do DOP (caderno principal, sem suplemento)."""
    destino = destino or DESTINO
    with requests.Session() as s:
        itens = _dop_pegar_itens_home(s)
        item = next((i for i in itens if i.get("suplemento") == ""), None)
        if not item:
            raise RuntimeError("Nenhuma edição com suplemento vazio encontrada na home.")
        data_str = datetime.date.today().strftime("%Y_%m_%d")
        return _dop_baixar_pdf(s, item["id"], f"DOP_{data_str}.pdf", destino)

def baixar_dop_por_data(data_iso: str, destino=None):
    """Tenta baixar o DOP de uma data específica (YYYY-MM-DD).

    Estratégia: procura a data em DADOS_ULTIMA_DATA (campos comuns: data,
    data_publicacao, data_edicao). Se não achar, levanta erro com instrução
    pro usuário inspecionar o portal.
    """
    destino = destino or DESTINO
    with requests.Session() as s:
        itens = _dop_pegar_itens_home(s)
        for item in itens:
            for key in ("data", "data_publicacao", "data_edicao"):
                if str(item.get(key, "")).startswith(data_iso) and item.get("suplemento") == "":
                    nome = f"DOP_{data_iso.replace('-', '_')}.pdf"
                    return _dop_baixar_pdf(s, item["id"], nome, destino)
        raise RuntimeError(
            f"DOP {data_iso} não está exposto em DADOS_ULTIMA_DATA. "
            "Pra datas mais antigas é preciso descobrir o endpoint de arquivo do portal. "
            f"Sugestão: abra {DOP_BASE} no navegador, procure a edição da data desejada "
            "e anote o link de download."
        )

def baixar_dop_por_edicao(numero: int, destino=None):
    """Tenta baixar o DOP pelo número da edição (procura na home)."""
    destino = destino or DESTINO
    with requests.Session() as s:
        itens = _dop_pegar_itens_home(s)
        for item in itens:
            for key in ("numero", "edicao", "numero_edicao"):
                if str(item.get(key, "")) == str(numero) and item.get("suplemento") == "":
                    nome = f"DOP_edicao_{numero}.pdf"
                    return _dop_baixar_pdf(s, item["id"], nome, destino)
        raise RuntimeError(
            f"DOP edição {numero} não está exposta em DADOS_ULTIMA_DATA. "
            "Veja a mensagem de baixar_dop_por_data."
        )

def baixar_dop_range(inicio, fim, tipo: str, destino=None):
    """Baixa um intervalo de DOPs. tipo='data' (inicio/fim YYYY-MM-DD) ou 'edicao' (inteiros)."""
    destino = destino or DESTINO
    saidas = []
    if tipo == "data":
        d_ini = datetime.date.fromisoformat(str(inicio))
        d_fim = datetime.date.fromisoformat(str(fim))
        d = d_ini
        while d <= d_fim:
            try:
                saidas.append(baixar_dop_por_data(d.isoformat(), destino))
            except Exception as e:
                print(f"[DOP] aviso ({d}): {e}")
            d += datetime.timedelta(days=1)
    elif tipo == "edicao":
        for n in range(int(inicio), int(fim) + 1):
            try:
                saidas.append(baixar_dop_por_edicao(n, destino))
            except Exception as e:
                print(f"[DOP] aviso (edicao {n}): {e}")
    else:
        raise ValueError("tipo deve ser 'data' ou 'edicao'")
    return saidas


## 4. DCM — Diário da Câmara Municipal

Fonte: <https://dcmdigital.camara.rj.gov.br>

A home renderiza dinamicamente, então usamos **Selenium** em modo headless para esperar o JavaScript carregar. Para edição específica, o site tem formulário nativo (ano em `<select>` + número em `<input>`).


In [ ]:
DCM_BASE = "https://dcmdigital.camara.rj.gov.br/"
DCM_UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
          "(KHTML, like Gecko) Chrome/120.0 Safari/537.36")
DCM_REQ_HEADERS = {"User-Agent": DCM_UA, "Referer": DCM_BASE,
                   "Accept": "application/pdf,*/*"}

def _dcm_novo_driver():
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options as ChromeOptions
    opts = ChromeOptions()
    for arg in ("--headless", "--no-sandbox", "--disable-dev-shm-usage",
                "--disable-gpu", "--window-size=1920,1080"):
        opts.add_argument(arg)
    opts.add_argument(f"--user-agent={DCM_UA}")
    return webdriver.Chrome(options=opts)

def _dcm_baixar_urls(urls, prefixo, destino):
    """Baixa cada URL como caderno parcial e une em um único PDF final."""
    parciais = []
    with requests.Session() as s:
        s.headers.update(DCM_REQ_HEADERS)
        for i, url in enumerate(urls, 1):
            try:
                r = s.get(url, timeout=120, verify=False)
                r.raise_for_status()
            except requests.exceptions.RequestException as e:
                print(f"[DCM] erro baixando {url}: {e}")
                continue
            if "application/pdf" not in r.headers.get("Content-Type", "").lower():
                print(f"[DCM] aviso: {url} não retornou PDF, pulando.")
                continue
            tmp = os.path.join(destino, f"_tmp_{prefixo}_{i}.pdf")
            with open(tmp, "wb") as f:
                f.write(r.content)
            parciais.append(tmp)
    if not parciais:
        raise RuntimeError("Nenhum caderno baixado com sucesso.")
    if len(parciais) == 1:
        final = os.path.join(destino, f"{prefixo}.pdf")
        shutil.move(parciais[0], final)
    else:
        final = os.path.join(destino, f"{prefixo}_unificado.pdf")
        merger = PdfMerger()
        try:
            for p in parciais:
                merger.append(p)
            merger.write(final)
        finally:
            merger.close()
        for p in parciais:
            try: os.remove(p)
            except OSError: pass
    print(f"[DCM] OK {final}")
    return final

def baixar_dcm_hoje(destino=None):
    """Baixa a última edição publicada do DCM (todos os cadernos do dia, unificados)."""
    destino = destino or DESTINO
    driver = _dcm_novo_driver()
    try:
        driver.get(DCM_BASE)
        time.sleep(5)
        html = driver.page_source
    finally:
        driver.quit()

    m = re.search(
        r'<span id="edAtual">\s*Última Edição:\s*(\d+)\s*</span>\s*<span>\s*-\s*(\d{2}/\d{2}/\d{4})',
        html
    )
    if not m:
        raise RuntimeError("Não consegui extrair número/data da edição mais recente.")
    edicao = m.group(1)
    data_str = m.group(2).replace("/", "_")
    print(f"[DCM] última edição: {edicao} ({data_str})")

    m_unico = re.search(
        r'<!-- Só tem 01 Caderno -->.*?<a href="(https://dcmdigital\.camara\.rj\.gov\.br/download/[^"]+)"',
        html, re.DOTALL
    )
    if m_unico:
        urls = [m_unico.group(1).strip()]
    else:
        ids = re.findall(
            r'class="btn btn-sm btn-primary btn-busca buscaDcmDoDia".+?arg="(\w+)"',
            html, re.DOTALL
        )
        if not ids:
            raise RuntimeError("Não encontrei links de caderno na home do DCM.")
        urls = [urljoin(DCM_BASE, f"download/{a}") for a in ids]

    return _dcm_baixar_urls(urls, f"DCM_{data_str}_ed{edicao}", destino)

def baixar_dcm_edicao(ano: int, numero: int, destino=None):
    """Baixa o DCM de uma edição específica (ano + número)."""
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import Select, WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.common.exceptions import TimeoutException
    destino = destino or DESTINO

    driver = _dcm_novo_driver()
    urls = []
    try:
        driver.get(DCM_BASE)
        wait = WebDriverWait(driver, 40)
        wait.until(EC.visibility_of_element_located((By.ID, "yDcm2")))
        time.sleep(2)
        Select(driver.find_element(By.ID, "yDcm2")).select_by_value(str(ano))
        time.sleep(1)
        inp = wait.until(EC.visibility_of_element_located((By.ID, "inputEdicao")))
        inp.clear()
        inp.send_keys(str(numero))
        wait.until(EC.element_to_be_clickable((By.ID, "btnEd"))).click()
        wait.until(EC.visibility_of_element_located((By.ID, "corpoModal")))
        time.sleep(2)
        try:
            figs = wait.until(EC.presence_of_all_elements_located(
                (By.XPATH, "//div[@id='corpoModal']/figure")))
        except TimeoutException:
            figs = []
        for fig in figs:
            try:
                link = fig.find_element(By.XPATH, ".//a[starts-with(@href, '/download/')]")
                href = link.get_attribute("href")
                if href:
                    urls.append(urljoin(DCM_BASE, href))
            except Exception:
                continue
    finally:
        driver.quit()

    if not urls:
        raise RuntimeError(f"DCM edição {numero}/{ano}: nenhum caderno encontrado.")
    return _dcm_baixar_urls(urls, f"DCM_ed{numero}_ano{ano}", destino)

def baixar_dcm_range(ano: int, inicio: int, fim: int, destino=None):
    """Baixa um intervalo de edições do DCM (mesmo ano)."""
    destino = destino or DESTINO
    saidas = []
    for n in range(int(inicio), int(fim) + 1):
        try:
            saidas.append(baixar_dcm_edicao(ano, n, destino))
        except Exception as e:
            print(f"[DCM] aviso (edicao {n}): {e}")
    return saidas


## 5. Runner

Interpreta as variáveis do topo e dispara as funções certas.

In [ ]:
def rodar():
    saidas = []
    if FONTE in ("dop", "ambos"):
        try:
            if MODO == "hoje":
                saidas.append(baixar_dop_hoje(DESTINO))
            elif MODO == "edicao":
                if EDICAO_DATA:
                    saidas.append(baixar_dop_por_data(EDICAO_DATA, DESTINO))
                else:
                    saidas.append(baixar_dop_por_edicao(EDICAO_NUMERO, DESTINO))
            elif MODO == "range":
                saidas += baixar_dop_range(RANGE_INICIO, RANGE_FIM, RANGE_TIPO, DESTINO)
            else:
                print(f"[DOP] MODO desconhecido: {MODO}")
        except Exception as e:
            print(f"[DOP] falhou: {e}")

    if FONTE in ("dcm", "ambos"):
        try:
            if MODO == "hoje":
                saidas.append(baixar_dcm_hoje(DESTINO))
            elif MODO == "edicao":
                saidas.append(baixar_dcm_edicao(EDICAO_ANO, EDICAO_NUMERO, DESTINO))
            elif MODO == "range":
                if RANGE_TIPO != "edicao":
                    print("[DCM] RANGE_TIPO='data' não se aplica ao DCM, pulando.")
                else:
                    saidas += baixar_dcm_range(EDICAO_ANO, RANGE_INICIO, RANGE_FIM, DESTINO)
            else:
                print(f"[DCM] MODO desconhecido: {MODO}")
        except Exception as e:
            print(f"[DCM] falhou: {e}")

    return [s for s in saidas if s]

pdfs_baixados = rodar()
print(f"\n{len(pdfs_baixados)} PDF(s) baixado(s).")


## 6. Resultado

Resumo dos arquivos baixados nesta execução.

In [ ]:
import pandas as pd

linhas = []
for p in pdfs_baixados:
    try:
        n_paginas = len(PdfReader(p).pages)
    except Exception:
        n_paginas = "?"
    linhas.append({
        "arquivo":   os.path.basename(p),
        "tamanho_MB": round(os.path.getsize(p) / 1024 / 1024, 2),
        "paginas":   n_paginas,
        "modificado": datetime.datetime.fromtimestamp(
            os.path.getmtime(p)).strftime("%Y-%m-%d %H:%M"),
    })

if linhas:
    df = pd.DataFrame(linhas)
    try:
        from IPython.display import display
        display(df)
    except ImportError:
        print(df.to_string(index=False))
else:
    print("Nenhum PDF foi baixado nesta execução.")
